# Traditional ML Experimentation with Data Drift Monitoring


## Overview

This notebook demonstrates an end-to-end machine learning workflow with Amazon SageMaker AI that includes:

- **Model Training**: Train an XGBoost model using SageMaker ModelTrainer (SDK v3)
- **MLflow Integration**: Track experiments and log artifacts to SageMaker AI MLflow App
- **Batch Transform**: Run batch inference on production data
- **Data Drift Monitoring**: Use Evidently (open-source) to detect data drift
- **Drift Metrics in MLflow**: Log drift metrics and reports to MLflow for tracking

### Business Problem

Predict whether a bank customer will subscribe to a term deposit based on:
- Demographics (age, job, marital status, education)
- Financial information (credit default, housing loan, personal loan)
- Campaign data (contact type, month, day of week, duration)
- Economic indicators (employment rate, consumer price index, etc.)

### Key Technologies

- **SageMaker Python SDK v3**: Simplified training and deployment APIs
- **SageMaker AI MLflow**: Fully managed experiment tracking and model registry
- **Evidently**: Open-source library for ML model monitoring and data drift detection
- **SageMaker Batch Transform**: Scalable batch inference for production data

---

<div class="alert alert-warning" role="alert">
<h3>⚠️ SIMULATED PRODUCTION DATA WITH ARTIFICIAL DRIFT</h3>
<p><strong>Purpose:</strong> Use the notebook to demonstrate solutions drift detection capabilities, demos and not for production deployments</p>

## Section 1: Prerequisites & Setup

Install required packages and import libraries.

In [ ]:
# Install required packages
# This may take 1-2 minutes on first run
%pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
%pip install --no-cache-dir sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops 'sagemaker>=3' mlflow==3.4.0 sagemaker-mlflow==0.2.0
%pip install -Uq "pandas" "scikit-learn" "xgboost" "evidently>=0.7.20" --no-cache-dir --ignore-installed

# Restart kernel to pick up updated packages
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
# Import required libraries
import boto3
import sagemaker
import mlflow
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import zipfile
import os
import json
import time
from datetime import datetime

# SageMaker pysdk v3 imports
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.core.training.configs import SourceCode, InputData, Compute
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.builder.schema_builder import SchemaBuilder
from sagemaker.core.transformer import Transformer

# Evidently imports (v0.7.20+) - Using correct preset names
from evidently import Report, Dataset, DataDefinition, BinaryClassification
from evidently.presets import DataDriftPreset, DataSummaryPreset, ClassificationPreset
from evidently.metrics import *

print(f'MLflow version: {mlflow.__version__}')

### Restore Variables from Lab 4.3

Load `X_test` and `model_data_s3` that were saved in the previous notebook.

In [ ]:
# Restore X_train, X_test, y_test, model_data_s3, and training_job_name saved from lab_4.3
import joblib

X_train = joblib.load('data/X_train.joblib')
X_test = joblib.load('data/X_test.joblib')
y_test = joblib.load('data/y_test.joblib')

with open('data/shared_vars.json', 'r') as f:
    shared_vars = json.load(f)

model_data_s3 = shared_vars['model_data_s3']
training_job_name = shared_vars['training_job_name']

print(f'✓ Restored X_train: {X_train.shape}')
print(f'✓ Restored X_test: {X_test.shape}')
print(f'✓ Restored y_test: {y_test.shape}')
print(f'✓ Restored model_data_s3: {model_data_s3}')
print(f'✓ Restored training_job_name: {training_job_name}')

# Save baseline (reference) data for drift monitoring - use training data features
X_train.to_csv('data/baseline.csv', index=False, header=True)



In [ ]:
# Initialize SageMaker session and get AWS configuration
sagemaker_session = Session()
region = sagemaker_session.boto_region_name
role = get_execution_role()
bucket = sagemaker_session.default_bucket()
prefix = 'test-monitoring-pipeline'

print('AWS Configuration:')
print(f'  Region: {region}')
print(f'  S3 Bucket: {bucket}')
print(f'  IAM Role: {role}')
print(f'  Data Prefix: {prefix}')

---

## Section 2: SageMaker AI MLflow App Configuration

### What is SageMaker AI MLflow App?

A fully managed service that provides:
- **Experiment Tracking**: Log parameters, metrics, and artifacts
- **Model Registry**: Version and manage models
- **SageMaker Integration**: Automatic registration to SageMaker AI Model Registry
- **Reproducibility**: Track code versions and dependencies
- **Collaboration**: Share experiments across teams

### Instructions

1. In SageMaker Studio, navigate to the left sidebar
2. Click on "MLflow" panel under "Applications"
3. Find your MLflow app (usually named "DefaultMLFlowApp")
4. Copy the ARN if you need to use a specific app

In [ ]:
# Configure MLflow app connection
mlflow_app_name = 'bank-classification'

# Get MLflow app ARN
sm_client = boto3.client('sagemaker', region_name=region)
mlflow_list = sm_client.list_mlflow_apps()

print(f'Found {len(mlflow_list["Summaries"])} MLflow app(s) in your account:')
for app in mlflow_list['Summaries']:
    print(f'  - {app["Name"]}')

# Find the specified MLflow app
mlflow_app_arn = None
for mlflow_app in mlflow_list['Summaries']:
    if mlflow_app['Name'] == mlflow_app_name:
        mlflow_app_arn = mlflow_app['Arn']
        break

if mlflow_app_arn:
    print(f'\n Using MLflow app: {mlflow_app_name}')
    print(f'  ARN: {mlflow_app_arn}')
else:
    raise ValueError(f'MLflow app "{mlflow_app_name}" not found. Please check the name or create one in SageMaker Studio.')

In [ ]:
# Set MLflow tracking URI and create/select experiment
mlflow.set_tracking_uri(mlflow_app_arn)
mlflow_experiment_name = 'test-monitor-pipeline'
mlflow_model_name = 'test-trad-xgb-mp'

try:
    # Try to create a new experiment
    experiment_id = mlflow.create_experiment(mlflow_experiment_name)
    print(f'✓ Created new MLflow experiment: {mlflow_experiment_name}')
    print(f'  Experiment ID: {experiment_id}')
except:
    # Experiment already exists, set it as active
    mlflow.set_experiment(mlflow_experiment_name)
    experiment = mlflow.get_experiment_by_name(mlflow_experiment_name)
    print(f'✓ Using existing MLflow experiment: {mlflow_experiment_name}')
    print(f'  Experiment ID: {experiment.experiment_id}')

print('\nYou can view your experiments in the SageMaker Studio MLflow App UI')

## Section 3: Batch Transform for Inference

Deploy the trained model for batch inference using SageMaker Batch Transform.

### Why Batch Transform?
- **Cost-effective**: No always-on endpoints
- **Scalable**: Process large datasets efficiently
- **Simple**: No infrastructure management
- **Perfect for**: Periodic predictions, large batch jobs, non-real-time use cases

In [ ]:
# Get inference image URI
inference_image = image_uris.retrieve(
    framework='xgboost',
    region=region,
    version='1.7-1',
    image_scope='inference'
)
print(f'XGBoost inference image: {inference_image}')

In [ ]:
# Create SageMaker Model
model_name = f'{mlflow_model_name}-{datetime.now().strftime("%Y%m%d%H%M%S")}'

create_model_response = sm_client.create_model(
    ModelName=model_name,
    PrimaryContainer={
        'Image': inference_image,
        'ModelDataUrl': model_data_s3
    },
    ExecutionRoleArn=role
)

print(f'Model created: {model_name}')

In [ ]:
# Prepare test data for batch transform (features only, no target)
test_features = X_test.copy()
test_features.to_csv('data/test_features.csv', index=False, header=False)
test_features.to_csv('data/test_features_headers.csv', index=False, header=True) # For monitoring job (Optional)
# Upload to S3
test_features_s3 = sagemaker_session.upload_data(
    'data/test_features.csv', 
    bucket, 
    f'{prefix}/batch-input'
)

print(f'Test features uploaded to: {test_features_s3}')

In [ ]:
test_features.shape

In [ ]:
# Create Transformer for batch inference
transformer = Transformer(
    model_name=model_name,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{bucket}/{prefix}/batch-output',
    sagemaker_session=sagemaker_session,
    assemble_with='Line',
    accept='text/csv'
)

In [ ]:
%%time
# Start batch transform job
# This will take approximately 5-10 minutes.
print('Starting batch transform job...')

transformer.transform(
    data=test_features_s3,
    content_type='text/csv',
    split_type='Line'
)

print('Waiting for batch transform to complete...')
transformer.wait()

print(f'Output location: {transformer.output_path}')

In [ ]:
# Download predictions
s3_client = boto3.client('s3')
output_key = f"{prefix}/batch-output/test_features.csv.out"

s3_client.download_file(bucket, output_key, 'data/predictions.csv')

# Load predictions
predictions = pd.read_csv('data/predictions.csv', header=None, names=['prediction_proba'])
predictions['prediction'] = (predictions['prediction_proba'] > 0.5).astype(int)

print(f'Predictions shape: {predictions.shape}')
print('\nSample predictions:')
print(predictions.head(10))

# Save predictions with features for drift analysis
test_with_predictions = test_features.copy()
test_with_predictions['prediction'] = predictions['prediction'].values
test_with_predictions.to_csv('data/test_with_predictions.csv', index=False)

In [ ]:
test_with_predictions

---

## Section 4: Data Drift Monitoring with Evidently

### What is Data Drift?

Data drift occurs when the statistical properties of input features change over time, which can degrade model performance.

### Why Monitor Drift?
- **Early Warning**: Detect issues before they impact business
- **Model Decay**: Understand when models need retraining
- **Data Quality**: Identify data pipeline issues
- **Compliance**: Track model performance for regulatory requirements

### Evidently Features
- **Open-source**: Free and customizable
- **Statistical Tests**: Multiple drift detection methods (KS test, Chi-square, etc.)
- **Visual Reports**: Interactive HTML reports
- **Flexible**: Works with any ML framework
- **MLflow Integration**: Log metrics and artifacts to MLflow

In [ ]:
# Load baseline (reference) data - training data distribution
reference_data = pd.read_csv('data/baseline.csv')

print(f'Reference (baseline) data shape: {reference_data.shape}')
print('\nReference data represents the training data distribution')
print('\nReference data statistics:')
print(reference_data.describe())

<div class="alert alert-warning" role="alert">
<h3>⚠️ SIMULATED PRODUCTION DATA WITH ARTIFICIAL DRIFT</h3>
<p><strong>Purpose:</strong> Introducing controlled data drift to demonstrate solutions drift detection capabilities</p>

<div class="alert alert-block alert-info">
<b>Important:</b>Introducing ARTIFICIAL data drift into the test set for demo purposes only. Do NOT use this in production. In a real scenario, this would be actual production data.
</div>

In [ ]:
# Simulate production data with drift
# In a real scenario, this would be actual production data
# Here we'll introduce some drift to demonstrate detection

production_data = test_features.copy()

# Introduce drift in some features to simulate real-world scenarios
# Feature 'age': shift distribution
if 'age' in production_data.columns:
    production_data['age'] = production_data['age'] + 2

# Feature 'duration': scale distribution
if 'duration' in production_data.columns:
    production_data['duration'] = production_data['duration'] * 1.15

# Feature 'campaign': add noise
if 'campaign' in production_data.columns:
    production_data['campaign'] = production_data['campaign'] + np.random.normal(0, 0.2, len(production_data))

# Align columns between reference and production data
# (handles cases where column names differ between notebooks, e.g. dots vs underscores)
common_cols = sorted(set(reference_data.columns) & set(production_data.columns))
if len(common_cols) < len(reference_data.columns):
    print(f'⚠️  Column mismatch detected. Aligning to {len(common_cols)} common columns.')
    print(f'   Reference-only: {sorted(set(reference_data.columns) - set(production_data.columns))}')
    print(f'   Production-only: {sorted(set(production_data.columns) - set(reference_data.columns))}')
    reference_data = reference_data[common_cols]
    production_data = production_data[common_cols]

print(f'Production data shape: {production_data.shape}')
print(f'Reference data shape: {reference_data.shape}')
print('\nProduction data statistics:')
print(production_data.describe())
print('\nSimulated drift introduced for demonstration')

In [ ]:
# Save artificial production data
production_data.to_csv('data/production_artificial_data.csv', index=False, header=True)

# Upload to S3
production_artificial_data_s3 = sagemaker_session.upload_data(
    'data/production_artificial_data.csv', 
    bucket, 
    f'{prefix}/batch-input'
)

print(f'artificial production data uploaded to: {production_artificial_data_s3}')

In [ ]:
# Helper utility to sanitize evidentyAI results DataDriftPreset for mlflow logging
import numpy as np

def log_metrics_from_dict(d):
    metrics = d.get("metrics", []) or []

    drifted_columns_logged = False

    for m in metrics:
        metric_name = m.get("metric_name", "")
        cfg = m.get("config", {}) or {}
        val = m.get("value")

        # 1) DriftedColumnsCount: always log
        if metric_name.startswith("DriftedColumnsCount"):
            drifted_columns_logged = True
            # value is a dict: {"count": ..., "share": ...}
            if isinstance(val, dict):
                count = float(val.get("count", 0.0))
                share = float(val.get("share", 0.0))
            else:
                # Fallback if structure changes
                count = float(val) if val is not None else 0.0
                share = 0.0

            mlflow.log_metric("DriftedColumnsCount.count", count)
            mlflow.log_metric("DriftedColumnsCount.share", share)
            continue

        # 2) ValueDrift: log only if value > threshold
        if metric_name.startswith("ValueDrift"):
            threshold = cfg.get("threshold")
            column = cfg.get("column")

            if threshold is None or column is None:
                continue

            try:
                numeric_val = float(val)
            except (TypeError, ValueError):
                continue

            if numeric_val > float(threshold):
                key = f"ValueDrift:{column}"
                mlflow.log_metric(key, numeric_val)

    # 3) If there was no DriftedColumnsCount metric at all, log 0
    if not drifted_columns_logged:
        mlflow.log_metric("DriftedColumnsCount", 0.0)

In [ ]:
# Create monitoring reports directory
os.makedirs('reports', exist_ok=True)

In [ ]:
# Create or reuse MLflow run for model monitoring
mlflow.set_experiment(mlflow_experiment_name)
timestamp_suffix = datetime.now().strftime('%Y%m%d_%H%M%S')

with mlflow.start_run(run_name=f"data_drift_quality_monitoring_{timestamp_suffix}") as drift_run:
    
    drift_run_id = drift_run.info.run_id
    
    # Log monitoring parameters
    mlflow.log_params({
        'reference_data_size': len(reference_data),
        'current_data_size': len(production_data),
        'monitoring_timestamp': datetime.now().isoformat(),
        'model_name': model_name,
        'training_job': training_job_name
    })

    # Wrap DataFrames in Evidently Dataset objects
    drift_data_definition = DataDefinition()
    reference_dataset = Dataset.from_pandas(reference_data, data_definition=drift_data_definition)
    production_dataset = Dataset.from_pandas(production_data, data_definition=drift_data_definition)
    
    # Create Data Drift Report using Evidently
    print('Creating Evidently data drift report...')
    
    data_drift_report = Report(metrics=[
        DataDriftPreset(),
    ])
    
    # Run the report
    data_drift_report_snapshot = data_drift_report.run(
        reference_data=reference_dataset,
        current_data=production_dataset
    )
    
    # Save report as HTML and json
    data_drift_report_filename = f'data_drift_report_{timestamp_suffix}'
    data_drift_report_snapshot.save_html(f'reports/{data_drift_report_filename}.html')
    data_drift_report_snapshot.save_json(f'reports/{data_drift_report_filename}.json')
    
    # Log report as MLflow artifact
    mlflow.log_artifact(f'reports/{data_drift_report_filename}.html', 'evidently_report_data_drift_html')
    mlflow.log_artifact(f'reports/{data_drift_report_filename}.json', 'evidently_report_data_drift_json')
    
    # Extract drift metrics from the report
    drift_results = data_drift_report_snapshot.dict()
    
    # Log summary drift metrics to MLflow
    log_metrics_from_dict(drift_results)

    # Create Data Summary Report (data quality metrics)
    print('Creating Evidently data summary report...')
    
    data_quality_report = Report(metrics=[
        DataSummaryPreset(),
    ])
    
    data_quality_report_snapshot = data_quality_report.run(
        reference_data=reference_data,
        current_data=production_data
    )
    
    # Save report
    data_quality_filename_html = f'data_quality_report_{timestamp_suffix}.html'
    data_quality_report_snapshot.save_html(f'reports/{data_quality_filename_html}')
    print(f'Data quality report saved to: {data_quality_filename_html}')
    
    # Log to MLflow
    mlflow.log_artifact(f'reports/{data_quality_filename_html}', 'evidently_report_data_quality')


In [ ]:
# Display the Evidently report inline
from IPython.display import IFrame

print('Displaying Evidently Data Drift Report:')
IFrame(src=f'reports/{data_drift_report_filename}.html', width=1000, height=800)

### View Drift Metrics in MLflow

To view your drift monitoring results in MLflow:

1. Open the SageMaker Studio MLflow UI
2. Navigate to the experiment: **bank-marketing-monitoring**
3. Find the run with name starting with **drift_monitoring_**
4. View:
   - **Metrics**: drift scores for each feature, dataset-level drift
   - **Artifacts**: HTML reports and JSON summaries
   - **Parameters**: monitoring configuration and timestamps

---

## Section 5: Binary Classification Performance Evaluation Model Quality with Evidently

Classification evaluation assesses how well your model performs on the prediction task by comparing predicted labels with actual ground truth labels.

### Key Classification Metrics

- **Accuracy**: Overall correctness of predictions
- **Precision**: Of all positive predictions, how many were actually positive (minimizes false positives)
- **Recall**: Of all actual positives, how many were correctly predicted (minimizes false negatives)
- **F1 Score**: Harmonic mean of precision and recall (balanced metric)
- **ROC-AUC**: Area under the ROC curve (measures discrimination ability)
- **Confusion Matrix**: Breakdown of true positives, true negatives, false positives, false negatives

### Why Evaluate Classification Performance?

- **Model Quality**: Understand model's predictive performance on test data
- **Business Metrics**: Align technical metrics with business objectives
- **Threshold Tuning**: Optimize decision threshold for your use case
- **Error Analysis**: Identify where the model makes mistakes

### Evidently Classification Features

- Comprehensive classification metrics (accuracy, precision, recall, F1, ROC-AUC)
- Confusion matrix visualization
- Classification quality by class
- Probability distribution analysis
- Prediction drift detection

In [ ]:
# Prepare data for classification evaluation
# We need both predictions and actual labels (y_test) for classification metrics

# Use y_test restored from lab_4.3 (matches X_test row count)
actual_labels = y_test.values

# Create evaluation dataset with predictions and actual labels
classification_eval_data = test_features.copy()
classification_eval_data['target'] = actual_labels
classification_eval_data['prediction'] = predictions['prediction'].values
classification_eval_data['prediction_proba'] = predictions['prediction_proba'].values

print(f'Classification evaluation dataset shape: {classification_eval_data.shape}')
print(f'\nActual labels distribution:')
print(pd.Series(actual_labels).value_counts())
print(f'\nPredicted labels distribution:')
print(pd.Series(predictions['prediction'].values).value_counts())

In [ ]:
# Helper utility to log mlflow metric 
import numpy as np

def log_evidently_classification_metrics_to_mlflow(metrics_dict):
    """Log Evidently classification metrics to MLflow using only metric prefix"""
    for metric in metrics_dict.get('metrics', []):
        # Extract only prefix before first '('
        metric_name_full = metric.get('metric_name', '')
        metric_name = metric_name_full.split('(')[0].strip()
        
        value = metric.get('value')
        
        if isinstance(value, dict):
            # Handle dict values like F1ByLabel, PrecisionByLabel, etc.
            for label, val in value.items():
                mlflow.log_metric(f"{metric_name}_label_{label}", float(val))
        else:
            # Handle scalar values (including np.float64)
            mlflow.log_metric(metric_name, float(value))

In [ ]:
# Start MLflow run for classification evaluation
#mlflow.set_experiment(mlflow_experiment_name)
#timestamp_suffix = datetime.now().strftime('%Y%m%d_%H%M%S')

with mlflow.start_run(run_name=f'model_quality_{timestamp_suffix}') as run:
    # Prepare data in the format Evidently expects
    eval_data = classification_eval_data.copy()

    # Ensure we have the necessary columns
    print(f"Evaluation dataset columns: {eval_data.columns.tolist()}")
    print(f"Evaluation dataset shape: {eval_data.shape}")
    
    # Create DataDefinition with BinaryClassification to tell Evidently which columns to use
    # This is REQUIRED for ClassificationPreset to work
    data_definition = DataDefinition(
        classification=[
            BinaryClassification(
                target='target',              # Column with actual labels
                prediction_labels='prediction', # Column with predicted labels
                pos_label=1,                  # Positive class label
            )
        ],
    )
    
    # Wrap the DataFrame in an Evidently Dataset
    eval_dataset = Dataset.from_pandas(
        eval_data,
        data_definition=data_definition,
    )
    
    # Create Classification Report using Evidently
    print('\nCreating Evidently classification performance report...')
    
    classification_report = Report(metrics=[
        ClassificationPreset(),
    ])
    
    # Run the report on test data with the Dataset object
    classification_report_snapshot = classification_report.run(
        reference_data=None,
        current_data=eval_dataset,
    )
    
    # Save report as HTML
    classification_report_filename = f'classification_report_{timestamp_suffix}'
    classification_report_snapshot.save_html(f'data/{classification_report_filename}.html')
    print(f'Classification report saved to: data/{classification_report_filename}.html')
    
    # Save report as JSON for programmatic access
    classification_report_snapshot.save_json(f'data/{classification_report_filename}.json')
    print(f'Classification report saved to: data/{classification_report_filename}.json')
    
    # Log the classification report artifacts to MLflow
    mlflow.log_artifact(f'data/{classification_report_filename}.html', 'evidently_classification_report_html')
    mlflow.log_artifact(f'data/{classification_report_filename}.json', 'evidently_classification_report_json')
    
    # Extract metrics from the report
    classification_report_dict = classification_report_snapshot.dict()
    # Log all Evidently model binary classification metrics to MLflow
    log_evidently_classification_metrics_to_mlflow(classification_report_dict)


### Classification Evaluation Summary

The classification report provides:

1. **Overall Performance Metrics**
   - Accuracy: Percentage of correct predictions
   - Precision: Quality of positive predictions
   - Recall: Coverage of actual positive cases
   - F1 Score: Balance between precision and recall
   - ROC-AUC: Discrimination ability across all thresholds

2. **Confusion Matrix**
   - True Positives (TP): Correctly predicted positive cases
   - True Negatives (TN): Correctly predicted negative cases
   - False Positives (FP): Incorrectly predicted as positive (Type I error)
   - False Negatives (FN): Incorrectly predicted as negative (Type II error)

3. **Class-Level Analysis**
   - Performance breakdown by class (0: No subscription, 1: Subscription)
   - Support: Number of samples per class
   - Per-class precision and recall

4. **Probability Calibration**
   - Distribution of predicted probabilities
   - Calibration curves showing reliability of probability estimates

### Business Impact

For the bank marketing use case:
- **High Precision**: Minimizes wasted effort on customers unlikely to subscribe
- **High Recall**: Maximizes capture of potential subscribers
- **F1 Score**: Balances both objectives for optimal campaign efficiency
- **Threshold Tuning**: Adjust decision boundary based on business costs/benefits

### Next Steps

- Review confusion matrix to understand error patterns
- Analyze false positives and false negatives for business impact
- Consider threshold adjustment if precision/recall tradeoff needs optimization
- Compare performance across different time periods or data segments

---

## Section 6: Comprehensive Monitoring Report (Optional)

Create a combined report with drift detection and data quality analysis.

In [ ]:
# Create comprehensive monitoring report
with mlflow.start_run(run_name=f"comprehensive_monitoring_{timestamp_suffix}") as comp_run:
    
    print('Creating comprehensive monitoring report...')
    
    comprehensive_report = Report(metrics=[
        DataDriftPreset(),
        DataSummaryPreset(),
    ])
    
    comprehensive_report_snapshot = comprehensive_report.run(
        reference_data=reference_data,
        current_data=production_data
    )
    
    # Save comprehensive report
    comp_filename = f'comprehensive_monitoring_{datetime.now().strftime("%Y%m%d_%H%M%S")}.html'
    comprehensive_report_snapshot.save_html(comp_filename)
    print(f'✓ Comprehensive report saved to: {comp_filename}')
    
    # Log to MLflow
    mlflow.log_artifact(comp_filename, 'evidently_comprehensive_report')
    
    # Upload to S3
    comp_s3_key = f"{prefix}/evidently-reports/{comp_filename}"
    s3_client.upload_file(comp_filename, bucket, comp_s3_key)
    
    mlflow.log_param('comprehensive_report_s3', f's3://{bucket}/{comp_s3_key}')
    
    print(f'\n✓ Comprehensive monitoring report completed')
    print(f'Report location: s3://{bucket}/{comp_s3_key}')

---

## Section 7: Summary and Next Steps

### What We Accomplished

1. **Model Training**: Trained XGBoost model using SageMaker SDK v3 with MLflow tracking
2. **Experiment Tracking**: Logged all parameters, metrics, and artifacts to SageMaker AI MLflow
3. **Batch Inference**: Deployed model using SageMaker Batch Transform for cost-effective predictions
4. **Classification Evaluation**: Evaluated binary classification performance using Evidently's ClassificationPreset with comprehensive metrics (accuracy, precision, recall, F1, ROC-AUC, confusion matrix)
5. **Drift Detection**: Used Evidently to detect data drift between training and production data
6. **MLflow Integration**: Logged all classification metrics, drift metrics, and reports to MLflow for tracking and comparison
7. **Visual Reports**: Generated interactive HTML reports for classification performance, data drift, and quality analysis

### Key Benefits

| Feature | Benefit |
|---------|--------|
| **SageMaker SDK v3** | Simplified APIs, better integration |
| **MLflow Tracking** | Complete experiment lineage and reproducibility |
| **Batch Transform** | Cost-effective inference without endpoints |
| **Evidently Classification** | Comprehensive binary classification metrics and visualizations |
| **Evidently Drift Detection** | Open-source, flexible, comprehensive drift detection |
| **Integrated Monitoring** | All metrics in one place (MLflow) |

### Next Steps

1. **Automated Monitoring**:
   - Set up scheduled monitoring using AWS Lambda + EventBridge
   - Trigger batch transform jobs periodically
   - Run classification evaluation and drift detection automatically

2. **Alerting**:
   - Configure SNS notifications for drift alerts
   - Set drift thresholds (e.g., >30% features drifted)
   - Alert on classification performance degradation (e.g., F1 < threshold)
   - Integrate with monitoring dashboards

3. **Model Retraining**:
   - Trigger retraining when drift exceeds thresholds
   - Trigger retraining when classification performance degrades
   - Use SageMaker Pipelines for automated retraining
   - A/B test new models against current production

4. **Enhanced Monitoring**:
   - Monitor prediction distribution drift over time
   - Track business metrics alongside technical metrics
   - Analyze classification errors for pattern detection
   - Compare classification performance across data segments

5. **Dashboard**:
   - Create custom dashboard using Evidently UI components
   - Visualize classification metrics and drift trends over time
   - Compare multiple monitoring runs
   - Track model performance degradation

### Viewing Results

**In MLflow UI**:
- All experiments, metrics, and artifacts are available
- Compare multiple runs side-by-side
- Download HTML reports from artifacts
- View classification metrics alongside drift metrics
- The reports generated are Accessible in S3 at the MLflow artifact location.
- Model quality and data quality reports available for download from MLflow UI

---

## Section 8: Cleanup (Optional)

Clean up resources to avoid unnecessary costs.

In [ ]:
# Delete the SageMaker model
try:
    sm_client.delete_model(ModelName=model_name)
    print(f'✓ Deleted model: {model_name}')
except Exception as e:
    print(f'Error deleting model: {e}')

print('\nNote: MLflow runs and S3 data are preserved for historical analysis.')

---

## Additional Resources

### Documentation
- [SageMaker Python SDK v3](https://sagemaker.readthedocs.io/en/stable/)
- [SageMaker AI MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- [Evidently Documentation](https://docs.evidentlyai.com/)
- [SageMaker Batch Transform](https://docs.aws.amazon.com/sagemaker/latest/dg/batch-transform.html)

### GitHub Repositories
- [Evidently GitHub](https://github.com/evidentlyai/evidently)
- [SageMaker Examples](https://github.com/aws/amazon-sagemaker-examples)

---

**Congratulations!** You've completed the Traditional ML Experimentation with Monitoring notebook.

You now have:
- A trained model tracked in MLflow
- Batch inference capability
- Data drift monitoring with Evidently
- All metrics and reports in MLflow for analysis

Happy monitoring! 🎉